# B05d — Soft Computing: Bayesian Reasoning and Fuzzy Logic

**COMPSCI 713 — Week 5: Soft Computing**

---

## Overview

Classical logic is crisp: a statement is either True or False. Real-world AI must handle **uncertainty** and **vagueness**. Soft computing provides two complementary tools:

- **Bayesian reasoning:** principled probability theory for uncertain beliefs
- **Fuzzy logic:** handling vague, linguistic concepts like "tall" or "hot"

In this lesson you will:
- Apply Bayes' theorem to update beliefs with evidence
- Build a Naive Bayes classifier from scratch
- Understand fuzzy sets and membership functions
- Build a fuzzy inference system (FIS)
- Compare probabilistic vs fuzzy approaches to uncertainty

### COMPSCI 713 Alignment
- **Week 5 Monday:** Soft Computing (Bayesian reasoning, Fuzzy logic & control)

## Part 1: Bayesian Reasoning

### Bayes' Theorem

$$P(H|E) = \frac{P(E|H) \cdot P(H)}{P(E)}$$

- **P(H)** — Prior: probability of hypothesis before seeing evidence
- **P(E|H)** — Likelihood: probability of evidence given hypothesis is true
- **P(E)** — Marginal: total probability of evidence
- **P(H|E)** — Posterior: updated probability after seeing evidence

**Intuition:** We start with a prior belief, observe evidence, and update our belief.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# ── Classic Medical Test Example ───────────────────────────────────────────
# Disease prevalence: 1% of population
# Test sensitivity (true positive rate): 99%
# Test specificity (true negative rate): 95%

P_disease       = 0.01   # prior
P_no_disease    = 1 - P_disease
P_pos_given_dis = 0.99   # sensitivity
P_pos_given_no  = 0.05   # 1 - specificity (false positive rate)

# P(positive test) = total probability
P_positive = P_pos_given_dis * P_disease + P_pos_given_no * P_no_disease

# Bayes: P(disease | positive test)
P_disease_given_pos = (P_pos_given_dis * P_disease) / P_positive

print("=== Medical Test: Bayesian Update ===")
print(f"Prior P(disease):                {P_disease:.2%}")
print(f"P(positive test):                {P_positive:.2%}")
print(f"Posterior P(disease | +test):    {P_disease_given_pos:.2%}")
print()
print("Surprising? Even with a 99% accurate test, a positive result")
print("only means ~16.7% chance of disease when prevalence is 1%.")
print("This is the base rate fallacy — Bayesian reasoning prevents it.")

In [ ]:
# ── Sequential Bayesian Updates ────────────────────────────────────────────
# What if we run the test twice?

def bayes_update(prior, likelihood_pos, likelihood_neg):
    """Update prior with new evidence."""
    p_evidence = likelihood_pos * prior + likelihood_neg * (1 - prior)
    posterior = (likelihood_pos * prior) / p_evidence
    return posterior

prior = 0.01
posteriors = [prior]
labels = ['Prior']

for i in range(1, 4):
    prior = bayes_update(prior, P_pos_given_dis, P_pos_given_no)
    posteriors.append(prior)
    labels.append(f'After test {i} (+)')

plt.figure(figsize=(8, 4))
plt.bar(labels, posteriors, color=['steelblue'] + ['coral']*3)
plt.ylabel('P(disease)')
plt.title('Sequential Bayesian Updates (all tests positive)')
plt.ylim(0, 1)
for i, v in enumerate(posteriors):
    plt.text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## 2. Naive Bayes Classifier

Naive Bayes applies Bayes' theorem to classification by assuming features are **conditionally independent** given the class — the "naive" assumption.

$$P(C|x_1,...,x_n) \propto P(C) \prod_{i=1}^{n} P(x_i|C)$$

Despite the naive assumption, it works surprisingly well for text classification, spam filtering, and medical diagnosis.

In [ ]:
class NaiveBayesClassifier:
    """Gaussian Naive Bayes from scratch."""

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors  = {}
        self.means   = {}
        self.stds    = {}

        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = len(X_c) / len(X)
            self.means[c]  = X_c.mean(axis=0)
            self.stds[c]   = X_c.std(axis=0) + 1e-9  # avoid division by zero

    def _gaussian_pdf(self, x, mean, std):
        return (1 / (std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mean) / std) ** 2)

    def predict_proba(self, X):
        probs = []
        for x in X:
            class_probs = {}
            for c in self.classes:
                log_prior = np.log(self.priors[c])
                log_likelihood = np.sum(np.log(self._gaussian_pdf(x, self.means[c], self.stds[c])))
                class_probs[c] = log_prior + log_likelihood
            probs.append(class_probs)
        return probs

    def predict(self, X):
        return [max(p, key=p.get) for p in self.predict_proba(X)]


from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42)

nb = NaiveBayesClassifier()
nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

print(f"Naive Bayes accuracy: {accuracy_score(y_test, y_pred):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

## Part 2: Fuzzy Logic

### The Problem with Crisp Logic

Classical (crisp) logic: a person is either TALL or NOT TALL.
- 1.79m → NOT TALL
- 1.80m → TALL

This is unrealistic. **Fuzzy logic** (Zadeh, 1965) allows partial membership:
- 1.79m → 0.49 TALL
- 1.80m → 0.50 TALL
- 2.00m → 0.95 TALL

### Fuzzy Sets
A fuzzy set A is defined by a **membership function** μ_A(x) ∈ [0, 1].

In [ ]:
# ── Membership Functions ───────────────────────────────────────────────────

def trimf(x, a, b, c):
    """Triangular membership function."""
    return np.maximum(0, np.minimum((x-a)/(b-a+1e-9), (c-x)/(c-b+1e-9)))

def trapmf(x, a, b, c, d):
    """Trapezoidal membership function."""
    return np.maximum(0, np.minimum(
        np.minimum((x-a)/(b-a+1e-9), 1),
        (d-x)/(d-c+1e-9)
    ))

# Temperature fuzzy sets
temp = np.linspace(0, 100, 500)

cold   = trapmf(temp, 0, 0, 15, 25)
cool   = trimf(temp, 15, 25, 35)
warm   = trimf(temp, 25, 35, 45)
hot    = trapmf(temp, 35, 45, 100, 100)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(temp, cold,  'b-',  linewidth=2, label='COLD')
ax.plot(temp, cool,  'c-',  linewidth=2, label='COOL')
ax.plot(temp, warm,  'y-',  linewidth=2, label='WARM')
ax.plot(temp, hot,   'r-',  linewidth=2, label='HOT')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Membership degree μ')
ax.set_title('Fuzzy Sets for Temperature')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Example: 30°C
t = 30
idx = np.argmin(np.abs(temp - t))
print(f"At {t}°C:")
print(f"  μ(COLD) = {cold[idx]:.2f}")
print(f"  μ(COOL) = {cool[idx]:.2f}")
print(f"  μ(WARM) = {warm[idx]:.2f}")
print(f"  μ(HOT)  = {hot[idx]:.2f}")

## 3. Fuzzy Inference System (Mamdani)

A fuzzy inference system (FIS) applies fuzzy IF-THEN rules:

```
IF temperature is HOT AND humidity is HIGH THEN fan_speed is FAST
IF temperature is WARM AND humidity is MEDIUM THEN fan_speed is MEDIUM
IF temperature is COLD THEN fan_speed is SLOW
```

**Steps:**
1. **Fuzzification:** convert crisp inputs to fuzzy membership values
2. **Rule evaluation:** apply fuzzy rules (AND = min, OR = max)
3. **Aggregation:** combine all rule outputs
4. **Defuzzification:** convert fuzzy output back to crisp value (centroid method)

In [ ]:
# ── Fan Speed Controller (Fuzzy Inference System) ──────────────────────────

fan_speed = np.linspace(0, 100, 500)

# Fan speed membership functions
slow   = trapmf(fan_speed, 0, 0, 20, 40)
medium = trimf(fan_speed, 30, 50, 70)
fast   = trapmf(fan_speed, 60, 80, 100, 100)

def fuzzy_fan_controller(temperature, humidity):
    """Simple 2-input fuzzy fan speed controller."""

    # ── Fuzzification ──────────────────────────────────────────────────────
    t = np.array([temperature])
    h = np.array([humidity])

    # Temperature membership
    mu_cold = float(trapmf(t, 0, 0, 15, 25))
    mu_warm = float(trimf(t, 25, 35, 45))
    mu_hot  = float(trapmf(t, 35, 45, 100, 100))

    # Humidity membership
    mu_low  = float(trapmf(h, 0, 0, 30, 50))
    mu_med  = float(trimf(h, 30, 50, 70))
    mu_high = float(trapmf(h, 50, 70, 100, 100))

    # ── Rule Evaluation (AND = min) ────────────────────────────────────────
    rules = [
        ('slow',   min(mu_cold, mu_low)),
        ('slow',   mu_cold),
        ('medium', min(mu_warm, mu_med)),
        ('medium', min(mu_warm, mu_low)),
        ('fast',   min(mu_hot, mu_high)),
        ('fast',   mu_hot),
        ('medium', min(mu_hot, mu_low)),
    ]

    # ── Aggregation ────────────────────────────────────────────────────────
    agg_slow   = max(strength for label, strength in rules if label == 'slow')
    agg_medium = max(strength for label, strength in rules if label == 'medium')
    agg_fast   = max(strength for label, strength in rules if label == 'fast')

    # Clip each output MF at its activation level
    out_slow   = np.minimum(slow,   agg_slow)
    out_medium = np.minimum(medium, agg_medium)
    out_fast   = np.minimum(fast,   agg_fast)

    # Aggregate (union = max)
    aggregated = np.maximum(np.maximum(out_slow, out_medium), out_fast)

    # ── Defuzzification (centroid) ─────────────────────────────────────────
    if aggregated.sum() == 0:
        return 0, aggregated
    crisp_output = np.sum(fan_speed * aggregated) / np.sum(aggregated)
    return crisp_output, aggregated


# Test the controller
test_cases = [(20, 30), (35, 50), (45, 80), (10, 20)]
print(f"{'Temperature':>12} {'Humidity':>10} {'Fan Speed':>12}")
print('-' * 36)
for temp_val, hum_val in test_cases:
    speed, _ = fuzzy_fan_controller(temp_val, hum_val)
    print(f"{temp_val:>10}°C {hum_val:>9}% {speed:>11.1f}%")

In [ ]:
# Visualise the fuzzy output for one case
temp_val, hum_val = 40, 70
speed, aggregated = fuzzy_fan_controller(temp_val, hum_val)

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(fan_speed, aggregated, alpha=0.4, color='purple', label='Aggregated output')
ax.plot(fan_speed, slow,   'b--', alpha=0.5, label='SLOW')
ax.plot(fan_speed, medium, 'g--', alpha=0.5, label='MEDIUM')
ax.plot(fan_speed, fast,   'r--', alpha=0.5, label='FAST')
ax.axvline(speed, color='black', linewidth=2, linestyle='-', label=f'Crisp output: {speed:.1f}%')
ax.set_xlabel('Fan Speed (%)')
ax.set_ylabel('Membership degree')
ax.set_title(f'Fuzzy Inference: Temp={temp_val}°C, Humidity={hum_val}% → Fan={speed:.1f}%')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Bayesian vs Fuzzy: When to Use Which?

| Aspect | Bayesian | Fuzzy |
|---|---|---|
| **Uncertainty type** | Probabilistic (random events) | Vagueness (linguistic terms) |
| **Requires** | Prior probabilities, data | Expert-defined membership functions |
| **Output** | Probability distribution | Crisp value via defuzzification |
| **Best for** | Diagnosis, classification, filtering | Control systems, linguistic rules |
| **Examples** | Spam filter, medical diagnosis | Washing machine, AC controller, ABS brakes |

## 5. Summary

### Key Takeaways
- **Bayes' theorem** updates beliefs with evidence: posterior ∝ likelihood × prior
- **Naive Bayes** assumes feature independence — surprisingly effective for classification
- **Fuzzy sets** allow partial membership — handling vague linguistic concepts
- **Fuzzy inference systems** map crisp inputs to crisp outputs via linguistic rules
- Both approaches handle uncertainty, but in fundamentally different ways

### Next Steps
- **B04b** — Expert Systems (rule-based AI with certainty factors)
- **I02** — Regularization (probabilistic interpretation of L1/L2)
- **E12** — Federated Learning (privacy-preserving Bayesian updates)